# Teleportación en una red óptica pasiva cuántica, en términos qg (§157, §158)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Viny2030/qang/blob/main/notebooks/qang_qpon_teleport_es.ipynb)

Una QPON de bajada: Alice comparte cos α|00⟩ + sin α|11⟩ y envía un qubit (un fotón presente o ausente) por una fibra (transmisión T) y un divisor pasivo 1:N hasta Bob, así que el fotón llega con probabilidad η = T/N. Es la versión de variable discreta del estudio de variable continua de Yaghoobianzadeh y Salehi (Academia Quantum 2026, 10.20935/AcadQuant8460). Todo corre en simuladores locales de dispositivos IBM (sin cuenta, sin hardware), en unos 5 minutos.

## 1. Instalación

In [1]:
import importlib, subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "qang", "qiskit-aer", "qiskit-ibm-runtime", "scikit-learn"], check=False)
importlib.invalidate_caches()
try:
    import qang.qpon
    ok = hasattr(qang.qpon, "dual_rail_teleport_circuit")
except ImportError:
    ok = False
if not ok:  # the published qang predates these sections: install from GitHub
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--force-reinstall", "--no-deps",
                    "git+https://github.com/Viny2030/qang.git"], check=True)
    print("installed qang from GitHub; if an import fails below, restart the runtime once")
import qang
print("qang", qang.__version__)

qang 0.11.0


## 2. Configuración

In [2]:
import json, math, os, time, urllib.request
import numpy as np
import qiskit_ibm_runtime.fake_provider as fp
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator

T0 = time.time()


def run(circuits, backend_name="FakeFez", shots=4000, seed=1):
    """Transpile to a fake IBM device and sample with its noise model; returns the list of counts."""
    b = getattr(fp, backend_name)()
    ts = transpile(circuits, b, optimization_level=1, seed_transpiler=137)
    res = AerSimulator.from_backend(b).run(ts, shots=shots, seed_simulator=seed).result()
    return [res.get_counts(i) for i in range(len(circuits))]


def repo_json(name):
    local = os.path.join("..", "examples", "data", name)
    if os.path.exists(local):
        return json.load(open(local, encoding="utf-8"))
    url = f"https://raw.githubusercontent.com/Viny2030/qang/main/examples/data/{name}"
    return json.loads(urllib.request.urlopen(url).read().decode("utf-8"))

from qang import qpon as Q

## 3. El recurso óptimo no es un par de Bell (§157)

La teleportación estándar da F = (2f + 1)/3 con fracción singlete f = (cos α + √η sin α)² / (2(cos²α + κ sin²α)). El óptimo es tan α* = √η/κ, F* = (2 + η/κ)/3, y el recurso óptimo tiene qg_Z* = (κ² − η)/(κ² + η) en cada qubit: una medición en Z fija el diseño. κ = 1 sin asistencia; κ = η + 1 − T cuando los otros usuarios avisan que el fotón les llegó a ellos (protocolo asistido: un filtro de sector sobre su registro).

In [3]:
print("link (L km, N)  protocol   alpha*   F*      F(pi/4)  qg_Z*")
for L, N in ((0, 4), (0, 8), (20, 4), (20, 8)):
    for prot, assisted in (("non", False), ("assisted", True)):
        eta, kap = Q.link(L, N, assisted)
        a = Q.optimal_angle(eta, kap)
        print(f"({L:2d}, {N})        {prot:9s}  {a:.3f}   {Q.optimal_fidelity(eta, kap):.4f}  "
              f"{Q.average_fidelity(math.pi / 4, eta, kap):.4f}   {Q.optimal_qg_z(eta, kap):+.3f}")

link (L km, N)  protocol   alpha*   F*      F(pi/4)  qg_Z*
( 0, 4)        non        0.464   0.7500  0.7083   +0.600
( 0, 4)        assisted   1.107   1.0000  0.9333   -0.600
( 0, 8)        non        0.340   0.7083  0.6387   +0.778
( 0, 8)        assisted   1.231   1.0000  0.8762   -0.778
(20, 4)        non        0.306   0.6998  0.6217   +0.819
(20, 4)        assisted   0.423   0.7140  0.6724   +0.663
(20, 8)        non        0.219   0.6833  0.5827   +0.905
(20, 8)        assisted   0.330   0.6921  0.6352   +0.790


En FakeFez, enlace (L = 0 km, N = 8): F contra α para los dos protocolos. Publicado (3 dispositivos): el óptimo medido cae en α*, sin asistencia solo el recurso óptimo supera el límite clásico 2/3, y la asistencia gana siempre.

In [4]:
L, N = 0, 8
T = Q.fibre_transmission(L)
alphas = [0.2, 0.4, 0.6, math.pi / 4, 1.0, 1.2, 1.4]
counts = run([Q.teleport_circuit(a, T, N, k) for a in alphas for k in range(6)])
print("alpha   F non-assisted   F assisted   kept (assisted)")
for i, a in enumerate(alphas):
    cs = counts[6 * i: 6 * i + 6]
    r0, r1 = Q.fidelity_from_counts(cs, False), Q.fidelity_from_counts(cs, True)
    print(f"{a:.3f}   {r0['fidelity']:.4f}           {r1['fidelity']:.4f}       {r1['kept']:.3f}")
for prot, assisted in (("non", False), ("assisted", True)):
    eta, kap = Q.link(L, N, assisted)
    print(f"{prot}: alpha* = {Q.optimal_angle(eta, kap):.3f}, F* = {Q.optimal_fidelity(eta, kap):.4f}")

alpha   F non-assisted   F assisted   kept (assisted)
0.200   0.6914           0.7064       0.944
0.400   0.6991           0.7565       0.850
0.600   0.6778           0.8037       0.713
0.785   0.6316           0.8485       0.565
1.000   0.5602           0.9065       0.390
1.200   0.4963           0.9417       0.252
1.400   0.4325           0.8465       0.167
non: alpha* = 0.340, F* = 0.7083
assisted: alpha* = 1.231, F* = 1.0000


## 4. La fidelidad desde tres correladores qg, y dual-rail (§158)

**A.** F = (3 + qg_XX − qg_YY + qg_ZZ)/6: tres circuitos cortos sobre el recurso predicen la fidelidad teleportada (publicado: con error menor a 0.03 en todos los casos).

**B.** Dual-rail: un fotón en dos modos por qubit, todas las compuertas conservan el número de fotones, y toda pérdida deja vacío el par de Bob, así que el filtro qg (peso 1 por par) la detecta. Sin ruido la F filtrada es 1 a cualquier distancia, con fracción conservada η. Publicado: los falsos avisos (un fotón perdido más un error del dispositivo) hacen bajar la F filtrada con la distancia, pero a 20 km supera al óptimo sin ruido de un solo modo en dos de tres dispositivos.

In [5]:
L, N = 0, 4
T = Q.fibre_transmission(L)
for prot, assisted in (("non", False), ("assisted", True)):
    a = Q.optimal_angle(*Q.link(L, N, assisted))
    cs = run([Q.resource_circuit(a, T, N, b) for b in "XYZ"] + [Q.teleport_circuit(a, T, N, k) for k in range(6)])
    corr = [Q.correlator_from_counts(c, assisted)[0] for c in cs[:3]]
    print(f"{prot:9s}: F from correlators {Q.fidelity_from_correlators(*corr):.4f}, "
          f"teleported {Q.fidelity_from_counts(cs[3:], assisted)['fidelity']:.4f}, "
          f"F* {Q.optimal_fidelity(*Q.link(L, N, assisted)):.4f}")

print("dual rail: link   filtered F   raw F   kept   eta    single-rail F* (best protocol)")
for L, N in ((0, 4), (0, 8), (20, 4), (20, 8)):
    eta = Q.fibre_transmission(L) / N
    r = Q.dual_rail_fidelity_from_counts(run([Q.dual_rail_teleport_circuit(math.pi / 4, eta, k) for k in range(6)]))
    best = max(Q.optimal_fidelity(*Q.link(L, N, a)) for a in (False, True))
    print(f"           ({L:2d},{N})   {r['filtered']:.4f}       {r['raw']:.4f}  {r['kept']:.3f}  {eta:.3f}  {best:.4f}")
print(f"total time {time.time() - T0:.0f} s")

non      : F from correlators 0.7437, teleported 0.7372, F* 0.7500


assisted : F from correlators 0.9701, teleported 0.9573, F* 1.0000
dual rail: link   filtered F   raw F   kept   eta    single-rail F* (best protocol)


           ( 0,4)   0.8709       0.5934  0.242  0.250  1.0000


           ( 0,8)   0.7968       0.5451  0.146  0.125  1.0000


           (20,4)   0.7790       0.5361  0.125  0.100  0.7140


           (20,8)   0.7117       0.5187  0.088  0.050  0.6921
total time 52 s


## Resumen

* La regla de diseño de un recurso QPON es un valor qg: qg_Z* = (κ² − η)/(κ² + η). Sin asistencia el par de Bell puede quedar debajo del límite clásico donde el recurso óptimo no.
* La asistencia de los otros usuarios es un filtro de sector.
* La fidelidad teleportada es una lectura qg lineal del recurso.
* Dual-rail convierte toda pérdida en un evento detectado; el filtro mantiene alta la fidelidad y paga en tasa, hasta que los falsos avisos ponen un piso.

Detalles: `RESEARCH_NOTES.md` §157–§158, `examples/qpon_teleport_qg.py`, `examples/qpon_teleport_dual_rail_qg.py`.